# 每月调仓决策（QC 云端版）

**这是本地 `manual/monthly_rebalance.ipynb` 的 QC 云端等效版本**。

区别：
- 本地版用 yfinance（免费、快但数据有微小差异）
- QC 版用 QuantBook（Quandl/IEX 付费数据，跟 backtest 同源）

用途：
- **平时**用本地版（更快）
- **核对/sanity check** 时用这个 QC 版本（数据更标准）
- 如果某月本地版和这个版本的 top 4 picks 不一致，说明 yfinance 数据有 issue，要复查

操作流程：
1. 在 Inputs cell 填当前持仓 + 现金
2. Kernel -> Restart Kernel（必须！QC research env 模块会缓存）
3. Cell -> Run All
4. 最后一格输出交易清单

## 1. 配置 + Imports（不要改）

In [ ]:
from AlgorithmImports import *
from QuantConnect.Research import QuantBook

import numpy as np
import pandas as pd
from datetime import datetime, timedelta

qb = QuantBook()
pd.set_option('display.float_format', '{:+.4f}'.format)

# Strategy parameters (locked after diagnostic tests)
# IEFA 替代 EFA: 同发达国际市场, 0.07% vs 0.32% expense ratio
# GLDM 替代 GLD: 同实物黄金, 0.10% vs 0.40% expense ratio
UNIVERSE = ['SPY', 'QQQ', 'IEFA', 'TLT', 'GLDM',
            'VWO', 'HYG', 'VNQ', 'IEF', 'DBC', 'SOXX']
LONG_N = 4
RANK_WEIGHTS_N4 = [0.40, 0.30, 0.20, 0.10]

print(f'Strategy: U{len(UNIVERSE)} + residual_mom + n={LONG_N} + rank weighting')
print(f'Universe: {UNIVERSE}')

## 2. * Inputs *（**每月只改这一格**）

In [ ]:
# 当前持仓 - ticker -> 股数（整数）
# 没持有的 ETF 不要列；首次运行就留空 dict {}
current_holdings = {
    # 'SPY':  12,
    # 'TLT':  25,
    # 'GLDM': 90,
    # 'SOXX': 5,
}

# 当前可用现金（USD）
cash_balance = 23000.0

# As-of date - None = today，or 'YYYY-MM-DD' to backdate
as_of_date_str = None

if as_of_date_str is None:
    as_of_date = pd.Timestamp(qb.time.date())
else:
    as_of_date = pd.Timestamp(as_of_date_str)

print(f'As of: {as_of_date.date()}')
print(f'Cash:  ${cash_balance:,.2f}')
print(f'Current holdings: {dict(current_holdings) or "(empty)"}')

## 3. 拉数据（QC 云端）

In [ ]:
# Subscribe each ticker to QC daily data
symbols = {t: qb.add_equity(t, Resolution.DAILY).symbol for t in UNIVERSE}

fetch_start = as_of_date - timedelta(days=400)
fetch_end   = as_of_date + timedelta(days=1)

history = qb.history(list(symbols.values()), fetch_start, fetch_end, Resolution.DAILY)

# Pivot to per-ticker close price DataFrame
closes = pd.DataFrame()
for ticker, sym in symbols.items():
    try:
        closes[ticker] = history.loc[sym]['close']
    except KeyError:
        print(f'  WARN: no data for {ticker}')
        continue
closes = closes.dropna(how='all')

print(f'Data: {len(closes)} trading days from {closes.index.min().date()} to {closes.index.max().date()}')

latest_prices = closes.iloc[-1]
print(f'\nLatest QC prices (as of {closes.index[-1].date()}):')
for t in UNIVERSE:
    p = latest_prices.get(t)
    if pd.notna(p):
        print(f'  {t:>5}: ${p:>9.2f}')
    else:
        print(f'  {t:>5}: MISSING')

## 4. 计算 residual_mom 信号

对每个 ETF：剥除其相对 SPY 的 beta，残差累积回报作为信号。

In [ ]:
def residual_mom_latest(asset_px, market_px):
    asset_ret = asset_px.pct_change()
    mkt_ret = market_px.pct_change()
    aligned = pd.concat([asset_ret, mkt_ret], axis=1, keys=['a', 'm']).dropna()
    if len(aligned) < 252:
        return np.nan
    win = aligned.iloc[-252:-21]
    if len(win) < 100:
        return np.nan
    a, m = win['a'].values, win['m'].values
    cov = float(np.cov(a, m, ddof=0)[0, 1])
    var = float(np.var(m, ddof=0))
    beta = cov / var if var > 1e-12 else 1.0
    return float((a - beta * m).sum())

signals = {}
for t in UNIVERSE:
    if t not in closes.columns or closes[t].isna().all():
        print(f'  WARN: {t} no data, skipping')
        continue
    signals[t] = residual_mom_latest(closes[t], closes['SPY'])

sig_df = pd.DataFrame({
    'signal':     [signals.get(t) for t in UNIVERSE],
    'rank':       0,
    'positive':   [signals.get(t, np.nan) > 0 if not pd.isna(signals.get(t, np.nan)) else False for t in UNIVERSE],
    'last_price': [latest_prices.get(t) for t in UNIVERSE],
}, index=UNIVERSE).sort_values('signal', ascending=False, na_position='last')
sig_df['rank'] = range(1, len(sig_df) + 1)

print('Signal table (sorted descending):')
print(sig_df.to_string())

## 5. Top 4 + 仓位分配（rank weighting）

In [ ]:
positives = sig_df[sig_df['positive']].head(LONG_N)
k = len(positives)

if k == 0:
    print('[!]  没有正动量 ETF — 全部持现金。')
    targets_pct = {}
elif k < LONG_N:
    print(f'[!]  只有 {k}/{LONG_N} 个 ETF 正动量。{(1-k/LONG_N)*100:.0f}% 仓位留现金。')
    weights_top4 = [(LONG_N - i) for i in range(LONG_N)]
    raw = weights_top4[:k]
    sum_all = sum(weights_top4)
    targets_pct = {positives.index[i]: raw[i] / sum_all for i in range(k)}
else:
    targets_pct = {positives.index[i]: RANK_WEIGHTS_N4[i] for i in range(LONG_N)}

holdings_value = sum(shares * latest_prices.get(t, 0)
                     for t, shares in current_holdings.items()
                     if t in latest_prices and pd.notna(latest_prices.get(t)))
total_value = cash_balance + holdings_value
print(f'\n当前组合：持仓 ${holdings_value:,.0f} + 现金 ${cash_balance:,.0f} = ${total_value:,.0f}')

print(f'\n目标分配（rank weighting）：')
for t, w in targets_pct.items():
    target_dollars = total_value * w
    target_shares = int(round(target_dollars / latest_prices[t]))
    rank = positives.index.get_loc(t) + 1
    print(f'  {t:>5}  rank #{rank}  weight {w*100:>5.1f}%  -> ${target_dollars:>10,.0f} ≈ {target_shares:>4d} shares @ ${latest_prices[t]:.2f}')

## 6. 交易清单

In [ ]:
target_shares = {}
for t in UNIVERSE:
    if t in targets_pct and t in latest_prices and pd.notna(latest_prices[t]) and latest_prices[t] > 0:
        target_shares[t] = int(round(total_value * targets_pct[t] / latest_prices[t]))
    else:
        target_shares[t] = 0

trade_rows = []
for t in UNIVERSE:
    curr = current_holdings.get(t, 0)
    tgt = target_shares.get(t, 0)
    diff = tgt - curr
    if diff == 0 and curr == 0:
        continue
    if pd.isna(latest_prices.get(t)):
        continue
    action = 'HOLD' if diff == 0 else ('BUY' if diff > 0 else 'SELL')
    trade_rows.append({
        'action': action, 'ticker': t,
        'current': curr, 'target': tgt, 'delta': diff,
        'price': latest_prices[t],
        'amount': abs(diff) * latest_prices[t],
    })

trade_df = pd.DataFrame(trade_rows)
if trade_df.empty:
    print('[OK] 当前持仓已与目标一致，本月无需交易。')
else:
    trade_df = trade_df.sort_values(['action', 'amount'], ascending=[True, False])
    sells = trade_df[trade_df.action == 'SELL']
    buys  = trade_df[trade_df.action == 'BUY']
    holds = trade_df[trade_df.action == 'HOLD']

    print('=' * 64)
    print(f'   交易清单（{as_of_date.date()}, QC data）')
    print('=' * 64)
    if not sells.empty:
        print('\n-> 先 SELL:')
        for _, r in sells.iterrows():
            print(f"   SELL  {r['ticker']:>5}  {r['delta']:>+5d} shares  @ ~${r['price']:>8.2f}  = ${r['amount']:>10,.0f}")
    if not buys.empty:
        print('\n-> 再 BUY:')
        for _, r in buys.iterrows():
            print(f"   BUY   {r['ticker']:>5}  {r['delta']:>+5d} shares  @ ~${r['price']:>8.2f}  = ${r['amount']:>10,.0f}")
    if not holds.empty:
        print('\n-> HOLD:')
        for _, r in holds.iterrows():
            print(f"   HOLD  {r['ticker']:>5}  {r['current']:>5d} shares  (no change)")

    total_sell = sells['amount'].sum() if not sells.empty else 0
    total_buy  = buys['amount'].sum()  if not buys.empty  else 0
    print(f"\n$  SELL ${total_sell:,.0f} - BUY ${total_buy:,.0f} = ${total_sell - total_buy:+,.0f}")
    print(f"$  交易后预计现金: ${cash_balance + total_sell - total_buy:,.0f}")
    print('=' * 64)

## 7. 跟本地 yfinance 版本对照

如果你已经在本地跑过 `manual/monthly_rebalance.ipynb`，把两边的 **top 4 picks** 对照：

| 一致情况 | 含义 | 行动 |
|---|---|---|
| 完全一致 | 数据等效 [OK] | 用本地 yfinance 版本即可 |
| 1 个不一致（在 rank 4 边缘） | 边界 case，数据微小差异 [!] | 任选一个执行，差距可忽略 |
| 2+ 个不一致 | 数据源有问题 ❌ | 用 QC 版本，调查 yfinance 数据 |

**注**：QC 数据通常比 yfinance 准但 QC 是付费 feed 转售，本地 yfinance 是免费抓取，两者偶有 split/dividend adjustment 时间差导致 close 价格略不同。Residual_mom 用 252 天求和，**这种微小差异基本被平均掉**。

经验：每年跑 1-2 次这个 QC 版本作为 sanity check 即可，**不需要每月都用**。